In [1]:
import polars as pl 
_=pl.Config.set_tbl_cols(100000)
_=pl.Config.set_tbl_rows(10000)
_=pl.Config.set_tbl_width_chars(10000)
_=pl.Config.set_fmt_str_lengths(10000)

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [2]:
example='/project/PlatigLab/data/RBP_ML/3_yogi_dataset_january_2025/all_events/HepG2_1000_num-peaks-no-kd.tsv.gz'

indices = pl.scan_csv(example, has_header=True, separator='\t',).select(pl.col('index')).rename({'index': 'graph_ID'}).collect(streaming=True)
assert indices['graph_ID'].is_unique().all(), "There are duplicate values in 'graph_ID'"


In [4]:
unique_SJ_combos = indices['graph_ID'].map_elements(
    lambda x: '_'.join(x.split('_')[0:8]), 
    return_dtype=str
).unique().sort()

unique_SJ_combos.head()

graph_ID
str
"""chr10_+_1000676_1000868_1000947_1001013_1005817_1005907"""
"""chr10_+_1000947_1001013_1005817_1005907_1007017_1007128"""
"""chr10_+_1000947_1001013_1007017_1007128_1008957_1009035"""
"""chr10_+_100347123_100347531_100348063_100348346_100360733_100364834"""
"""chr10_+_100348063_100348346_100352365_100352496_100354426_100354632"""
"""chr10_+_100348063_100348346_100352365_100352496_100360733_100364834"""
"""chr10_+_100348063_100348346_100354426_100354632_100360733_100364834"""
"""chr10_+_100352365_100352496_100354426_100354632_100356531_100356764"""
"""chr10_+_100352365_100352496_100354426_100354632_100360733_100364834"""


In [59]:
gtf = pl.read_csv("/project/PlatigLab/data/annotations/gencode.v29.primary_assembly.annotation_UCSC_names.gtf.gz", separator="\t", has_header=False, skip_rows=5)
gtf = gtf.filter(pl.col("column_3") == "exon")

gtf = gtf.with_columns([
    pl.col("column_9")
    .cast(pl.Utf8)
    .str.extract(r'exon_id "([^"]+)"', 1)
    .alias("exon_id")
])

assert gtf.select(pl.col("exon_id").str.starts_with("ENSE")).to_series().all(), "Not all exon_id values start with 'ENSE'"
gtf.shape
gtf.head()

(1263028, 10)

column_1,column_2,column_3,column_4,column_5,column_6,column_7,column_8,column_9,exon_id
str,str,str,i64,i64,str,str,str,str,str
"""chr1""","""HAVANA""","""exon""",11869,12227,""".""","""+""",""".""","""gene_id ""ENSG00000223972.5""; transcript_id ""ENST00000456328.2""; gene_type ""transcribed_unprocessed_pseudogene""; gene_name ""DDX11L1""; transcript_type ""processed_transcript""; transcript_name ""DDX11L1-202""; exon_number 1; exon_id ""ENSE00002234944.1""; level 2; transcript_support_level ""1""; tag ""basic""; havana_gene ""OTTHUMG00000000961.2""; havana_transcript ""OTTHUMT00000362751.1"";""","""ENSE00002234944.1"""
"""chr1""","""HAVANA""","""exon""",12613,12721,""".""","""+""",""".""","""gene_id ""ENSG00000223972.5""; transcript_id ""ENST00000456328.2""; gene_type ""transcribed_unprocessed_pseudogene""; gene_name ""DDX11L1""; transcript_type ""processed_transcript""; transcript_name ""DDX11L1-202""; exon_number 2; exon_id ""ENSE00003582793.1""; level 2; transcript_support_level ""1""; tag ""basic""; havana_gene ""OTTHUMG00000000961.2""; havana_transcript ""OTTHUMT00000362751.1"";""","""ENSE00003582793.1"""
"""chr1""","""HAVANA""","""exon""",13221,14409,""".""","""+""",""".""","""gene_id ""ENSG00000223972.5""; transcript_id ""ENST00000456328.2""; gene_type ""transcribed_unprocessed_pseudogene""; gene_name ""DDX11L1""; transcript_type ""processed_transcript""; transcript_name ""DDX11L1-202""; exon_number 3; exon_id ""ENSE00002312635.1""; level 2; transcript_support_level ""1""; tag ""basic""; havana_gene ""OTTHUMG00000000961.2""; havana_transcript ""OTTHUMT00000362751.1"";""","""ENSE00002312635.1"""
"""chr1""","""HAVANA""","""exon""",12010,12057,""".""","""+""",""".""","""gene_id ""ENSG00000223972.5""; transcript_id ""ENST00000450305.2""; gene_type ""transcribed_unprocessed_pseudogene""; gene_name ""DDX11L1""; transcript_type ""transcribed_unprocessed_pseudogene""; transcript_name ""DDX11L1-201""; exon_number 1; exon_id ""ENSE00001948541.1""; level 2; transcript_support_level ""NA""; ont ""PGO:0000005""; ont ""PGO:0000019""; tag ""basic""; havana_gene ""OTTHUMG00000000961.2""; havana_transcript ""OTTHUMT00000002844.2"";""","""ENSE00001948541.1"""
"""chr1""","""HAVANA""","""exon""",12179,12227,""".""","""+""",""".""","""gene_id ""ENSG00000223972.5""; transcript_id ""ENST00000450305.2""; gene_type ""transcribed_unprocessed_pseudogene""; gene_name ""DDX11L1""; transcript_type ""transcribed_unprocessed_pseudogene""; transcript_name ""DDX11L1-201""; exon_number 2; exon_id ""ENSE00001671638.2""; level 2; transcript_support_level ""NA""; ont ""PGO:0000005""; ont ""PGO:0000019""; tag ""basic""; havana_gene ""OTTHUMG00000000961.2""; havana_transcript ""OTTHUMT00000002844.2"";""","""ENSE00001671638.2"""


In [60]:
gtf = gtf.select([
    pl.col("column_1"),
    pl.col("column_4"),
    pl.col("column_5"),
    pl.col("column_3"),
    pl.col("exon_id"),
    pl.col("column_7")
])

gtf = gtf.unique()
gtf.head()

column_1,column_4,column_5,column_3,exon_id,column_7
str,i64,i64,str,str,str
"""chr8""",132751785,132751997,"""exon""","""ENSE00003591281.1""","""-"""
"""chr3""",113882752,113882832,"""exon""","""ENSE00003589886.1""","""+"""
"""chr9""",420949,421078,"""exon""","""ENSE00003557090.1""","""+"""
"""chrX""",136207016,136207190,"""exon""","""ENSE00000890595.1""","""+"""
"""chr5""",113101734,113101945,"""exon""","""ENSE00002025628.1""","""-"""


In [70]:
exon_counts = gtf.select("exon_id").to_series().value_counts().sort('count', descending=True)
exon_counts = exon_counts.filter(pl.col('count')>1)
exon_counts.head()

for exon_id in exon_counts["exon_id"]:
    subset = gtf.filter(pl.col("exon_id") == exon_id)
    assert subset.select(pl.col("column_1")).to_series().is_in(["chrX", "chrY"]).all(), f"Exon {exon_id} is not on chrX or chrY"

exon_id,count
str,u32
"""ENSE00001888780.1""",2
"""ENSE00003500487.1""",2
"""ENSE00001487873.1""",2
"""ENSE00001702996.1""",2
"""ENSE00001716328.2""",2
